# Plantas baixas com verificação determinística

Um modelo de linguagem propõe a planta de uma casa a partir de uma descrição em
português. Um verificador escrito em código confere a proposta contra regras de
projeto e escreve um relatório do que está errado. O relatório volta para o
modelo, que redesenha a planta inteira. E assim por diante, até passar.

A tese da oficina cabe numa frase: **gerar com o modelo, verificar com código.**

Rode as células de cima para baixo. Nada aqui precisa de terminal.

> Esta é uma ferramenta didática. Ela **não substitui projeto de profissional
> habilitado**, e as regras embutidas são uma simplificação.

## 1. Configuração

Três informações para falar com o modelo. Elas ficam só nesta sessão: nada é
gravado em arquivo, e some tudo quando você fechar o Codespace.

In [ ]:
import os
from getpass import getpass
from pathlib import Path

from floorplan_guardrails.generator import GeneratorError, generator_from_env
from floorplan_guardrails.loop import run_loop
from floorplan_guardrails.renderer import draw_history
from floorplan_guardrails.rules import load_rules
from floorplan_guardrails.runlog import RunLog

# O notebook acha a raiz do repositório subindo as pastas até encontrar o
# arquivo de regras, então funciona independente de onde o kernel foi aberto.
RAIZ = next(
    pasta
    for pasta in (Path.cwd(), *Path.cwd().parents)
    if (pasta / "config" / "rules.yaml").exists()
)
ARQUIVO_DE_REGRAS = RAIZ / "config" / "rules.yaml"

print("Repositório encontrado em:", RAIZ)

In [ ]:
def pedir(variavel: str, rotulo: str, secreto: bool = False) -> None:
    """Pede um valor e guarda só na sessão, sem gravar em lugar nenhum."""
    if os.environ.get(variavel):
        print(f"{rotulo}: já preenchido nesta sessão.")
        return

    valor = getpass(f"{rotulo}: ") if secreto else input(f"{rotulo}: ")
    os.environ[variavel] = valor.strip()


pedir("AZURE_OPENAI_ENDPOINT", "Endpoint do Azure")
pedir("AZURE_OPENAI_DEPLOYMENT", "Nome do deployment")
pedir("AZURE_OPENAI_API_KEY", "Chave de API", secreto=True)

# Esta variável precisa NÃO existir. O caminho /openai/v1 recusa o parâmetro
# api-version, e o cliente a lê do ambiente por conta própria: se ela estiver
# definida, toda chamada falha com erro 400.
os.environ.pop("AZURE_OPENAI_API_VERSION", None)

DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]

In [ ]:
try:
    gerador = generator_from_env()
    resposta = await gerador.check()
except GeneratorError as erro:
    print("A conexão NÃO funcionou.\n")
    print(erro)
else:
    print("Conexão funcionando. O modelo respondeu:", resposta)

In [ ]:
regras = load_rules(ARQUIVO_DE_REGRAS)


async def projetar(descricao: str, regras_em_uso) -> object:
    """Roda o laço inteiro e conta como foi.

    É aqui que o projeto acontece: o gerador propõe, o validador confere, e o
    relatório de violações volta para o gerador. O laço para quando a planta
    passa, ou quando acabam as tentativas.
    """
    execucao = await run_loop(
        descricao,
        gerador,
        regras_em_uso,
        log=RunLog.create(RAIZ / "runs"),
        deployment=DEPLOYMENT,
    )

    if execucao.approved:
        print(f"Aprovada na iteração {len(execucao.iterations)}.")
    else:
        print(f"Não convergiu em {len(execucao.iterations)} iterações.")

    for numero, iteracao in enumerate(execucao.iterations, 1):
        print(f"\nIteração {numero}: {len(iteracao.violations)} violações")
        for violacao in iteracao.violations:
            print("  -", violacao.message)

    return execucao

## 2. Nível 1 — a primeira casa

Escreva a casa que você quer em português comum, na célula abaixo. Depois rode
a célula seguinte e veja o laço trabalhar.

Cada iteração pode levar de dez a vinte segundos, então tenha paciência: o
modelo está redesenhando a casa inteira a cada tentativa.

In [ ]:
descricao = (
    "Uma casa pequena para um casal, com sala, cozinha, um quarto e um banheiro. "
    "A entrada dá para a sala."
)

In [ ]:
nivel1 = await projetar(descricao, regras)

In [ ]:
desenho1 = draw_history(nivel1.history)

Repare no que aconteceu. O modelo recebeu as convenções de desenho — onde fica a
origem, como se chamam as paredes, o que é uma planta coerente — mas **nunca
recebeu os números das regras**. Ele não sabe qual é a área mínima de um quarto
nem quanta janela um cômodo precisa ter.

A única forma de ele descobrir é errando e lendo o relatório. É por isso que a
primeira planta quase sempre reprova.

## 3. Nível 2 — provocando a reprovação

Agora o contrário: descrições feitas para dar errado. Escolha uma, rode, e leia
o relatório com calma. O que interessa aqui não é a planta final, é o que o
verificador conseguiu medir.

In [ ]:
PROVOCACOES = {
    "quarto minúsculo": (
        "Uma casa com um quarto bem pequeno, de uns dois metros por dois, "
        "mais sala, cozinha e banheiro."
    ),
    "banheiro sem janela": (
        "Uma casa com sala, cozinha, um quarto e um banheiro interno, "
        "sem nenhuma janela no banheiro."
    ),
    "cômodo sem porta": (
        "Uma casa com sala, cozinha e dois quartos, em que o segundo quarto "
        "não tem porta nenhuma."
    ),
    "casa grande demais": (
        "Uma casa com seis quartos, três banheiros, sala, cozinha e área de "
        "serviço, toda espremida num terreno de seis metros por seis."
    ),
    "casa que não fecha": (
        "Uma casa com quatro quartos, três banheiros, sala, cozinha, área de "
        "serviço e um corredor central."
    ),
}

for nome in PROVOCACOES:
    print("-", nome)

In [ ]:
# Troque o nome entre aspas por qualquer um dos listados acima.
escolhida = PROVOCACOES["quarto minúsculo"]

nivel2 = await projetar(escolhida, regras)

In [ ]:
desenho2 = draw_history(nivel2.history)

## 4. Nível 3 — mudando as regras

Até aqui o verificador cobrou o que está escrito em `config/rules.yaml`. Esse
arquivo é seu: mude um número e o que conta como casa aceitável muda junto.

Primeiro, veja o que a regra de área mínima diz hoje.

In [ ]:
regra = regras.rule("min_area")

print(regra.description)
print("Fonte:", regra.source)
print()
for tipo, valor in regra.values.items():
    print(f"  {tipo}: {valor}")

Agora abra `config/rules.yaml` no editor, ao lado deste notebook, e mude o valor
de `bedroom` dentro de `min_area` — experimente `14.0`, por exemplo. Salve o
arquivo e rode as duas células abaixo.

A mesma descrição do nível 1 vai passar por um verificador mais exigente.

In [ ]:
regras = load_rules(ARQUIVO_DE_REGRAS)

print("Mínimo para quartos agora:", regras.value("min_area", "bedroom"), "m²")

In [ ]:
nivel3 = await projetar(descricao, regras)

In [ ]:
desenho3 = draw_history(nivel3.history)

## 5. Fechamento

O que você viu:

- um modelo de linguagem propondo uma planta a partir de texto livre;
- um verificador em Python medindo essa planta e escrevendo, em português, o que
  está errado e por quê;
- o relatório voltando ao modelo até a planta passar;
- e, no nível 3, a regra mudando de lugar — porque quem decide o que é aceitável
  é o arquivo de regras, não o modelo.

O que a ferramenta **não** faz: mais de um pavimento, estrutura, escadas,
mobiliário, orientação solar, recuos do lote, desenho técnico executivo. Os
valores normativos aqui são provisórios e simplificados.

Ela serve para mostrar uma ideia: quando a saída de um modelo pode ser medida,
vale a pena medir — e devolver a medição para ele corrigir.

Cada execução ficou gravada em `runs/`, uma linha por iteração, caso você queira
olhar depois o que foi proposto e o que foi cobrado.